# ML-03 — Frame Your Lane as an ML Task

**Student:** Imran  
**Track:** Machine Learning  
**Lane:** Refresh / Content Opportunity Scoring  
**Notebook:** `w02_ml_task_framing.ipynb`

## Objective

The purpose of this notebook is to frame my selected content lane as a clear machine learning task. I will define the task type, target or proxy, success metric, unit of analysis, supported content action, and explain why machine learning is more suitable than a fixed rule.

## 1) My Lane as an ML Task

My selected lane is **Refresh / Content Opportunity Scoring**.

The primary machine learning task type is **scoring**. Each content page will receive a numerical refresh-opportunity score between 0 and 100. A higher score will indicate that a page should receive earlier human review.

The scores can then be used for **ranking**, with the highest-scoring pages appearing first in the review queue.

The output will help content editors decide which pages should be reviewed first, refreshed, expanded, protected, monitored, or investigated further.

The model will support editorial decisions. It will not automatically update, publish, remove, or prune content.

## 2) Target or Proxy

The ideal future target is:

`target_incremental_clicks_28d`

This target would represent the additional organic clicks received during the 28 days after a content refresh compared with the page's previous baseline.

However, this target is not currently available because it requires historical refresh actions and post-refresh performance data.

For the current framing task, I will create the following proxy:

`proxy_refresh_opportunity_score`

The proxy score will use available page-level signals such as impressions, click-through rate, average search position, and content staleness or last-update date.

A high proxy score means that a page has meaningful visibility but may still have room for improvement. It is an opportunity signal, not a guarantee that refreshing the page will improve performance.

## 3) Success Metric

The primary success metric will be **NDCG@10**.

NDCG@10 measures whether the pages with the highest real post-refresh gains appear near the top of the first ten recommendations.

This metric is appropriate because the main business action is to create a useful ranked review queue for content editors rather than only predicting an exact number.

Once post-refresh outcome data becomes available, the ranked recommendations can be compared with the actual incremental organic clicks achieved by the refreshed pages.

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for folder in [start, *start.parents]:
        if (folder / ".git").exists() or (folder / "work").exists():
            return folder
    return start

REPO_ROOT = find_repo_root(Path.cwd())

print("Current working directory:", Path.cwd())
print("Repository root:", REPO_ROOT)

## 4) The Unit of Analysis as a Real DataFrame

The required unit of analysis for this lane is:

> **One row represents one content page or URL.**

The following cells locate the starter dataset, load it as a pandas DataFrame, inspect its columns, and convert it into a page-level dataset when repeated page records are present.

In [ ]:
# Leave this as None for automatic file detection.
# To manually select a file, use a path such as:
# DATA_FILE = "data/starter_content_data.csv"

DATA_FILE = None

allowed_extensions = {".csv", ".parquet", ".json", ".jsonl", ".xlsx"}
ignored_parts = {".git", ".venv", "venv", "node_modules", "__pycache__"}

preferred_data_folders = [
    REPO_ROOT / "data",
    REPO_ROOT / "datasets",
    REPO_ROOT / "starter_data",
    REPO_ROOT / "work" / "data",
    REPO_ROOT / "resources" / "data",
]

existing_data_folders = [folder for folder in preferred_data_folders if folder.exists()]
search_locations = existing_data_folders if existing_data_folders else [REPO_ROOT]

data_files = []

for location in search_locations:
    for path in location.rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in allowed_extensions
            and not any(part in ignored_parts for part in path.parts)
        ):
            data_files.append(path)

data_files = list(dict.fromkeys(data_files))

if not data_files:
    raise FileNotFoundError(
        "No CSV, Parquet, JSON, JSONL, or Excel data file was found in the repository."
    )

def file_priority(path: Path) -> tuple:
    relative_name = str(path.relative_to(REPO_ROOT)).lower()
    keywords = ["starter", "content", "page", "url", "search", "gsc", "seo"]
    score = sum(keyword in relative_name for keyword in keywords)
    return (-score, len(relative_name))

data_files = sorted(data_files, key=file_priority)

print("Available data files:\n")
for index, path in enumerate(data_files[:20]):
    print(f"{index}: {path.relative_to(REPO_ROOT)}")

if DATA_FILE is None:
    selected_data_file = data_files[0]
else:
    selected_data_file = Path(DATA_FILE)
    if not selected_data_file.is_absolute():
        selected_data_file = REPO_ROOT / selected_data_file
    selected_data_file = selected_data_file.resolve()

print("\nSelected data file:")
print(selected_data_file.relative_to(REPO_ROOT))

In [ ]:
file_extension = selected_data_file.suffix.lower()

if file_extension == ".csv":
    df = pd.read_csv(selected_data_file)
elif file_extension == ".parquet":
    df = pd.read_parquet(selected_data_file)
elif file_extension == ".jsonl":
    df = pd.read_json(selected_data_file, lines=True)
elif file_extension == ".json":
    df = pd.read_json(selected_data_file)
elif file_extension == ".xlsx":
    df = pd.read_excel(selected_data_file)
else:
    raise ValueError(f"Unsupported file type: {file_extension}")

print("Dataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())
print("\nFirst five rows:")
display(df.head())

### Initial Data Observation

The starter data has been loaded successfully as a real DataFrame.

Before creating the proxy, I need to identify which columns represent the page, clicks, impressions, click-through rate, search position, and update date.

The code below searches for commonly used versions of these column names so that the notebook remains compatible with the starter data structure.

In [ ]:
def normalise_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

normalised_columns = {
    normalise_name(column): column
    for column in df.columns
}

def find_column(*candidates):
    for candidate in candidates:
        candidate_key = normalise_name(candidate)
        if candidate_key in normalised_columns:
            return normalised_columns[candidate_key]

    for candidate in candidates:
        candidate_key = normalise_name(candidate)
        for normalised, original in normalised_columns.items():
            if candidate_key in normalised or normalised in candidate_key:
                return original

    return None

page_col = find_column(
    "url", "page", "page_url", "landing_page",
    "content_url", "article_url", "path", "slug"
)

clicks_col = find_column("clicks", "organic_clicks", "search_clicks")
impressions_col = find_column("impressions", "organic_impressions", "search_impressions")
ctr_col = find_column("ctr", "click_through_rate", "clickthrough_rate")
position_col = find_column("position", "average_position", "avg_position", "search_position", "rank")
updated_col = find_column(
    "last_updated", "updated_at", "last_modified",
    "modified_at", "published_at", "publish_date", "date"
)

detected_columns = pd.DataFrame({
    "Required role": [
        "Page identifier",
        "Clicks",
        "Impressions",
        "CTR",
        "Search position",
        "Last update date",
    ],
    "Detected column": [
        page_col,
        clicks_col,
        impressions_col,
        ctr_col,
        position_col,
        updated_col,
    ],
})

display(detected_columns)

In [ ]:
def numeric_series(series: pd.Series, percentage: bool = False) -> pd.Series:
    text_values = series.astype(str).str.strip()
    contains_percentage = text_values.str.contains("%", regex=False).any()

    numeric_values = pd.to_numeric(
        text_values.str.replace("%", "", regex=False),
        errors="coerce",
    )

    if percentage and contains_percentage:
        numeric_values = numeric_values / 100

    return numeric_values

working_df = df.copy()

if page_col is None:
    working_df = (
        working_df
        .reset_index(drop=False)
        .rename(columns={"index": "content_item_id"})
    )
    page_col = "content_item_id"

for column in [clicks_col, impressions_col, position_col]:
    if column is not None:
        working_df[column] = numeric_series(working_df[column])

if ctr_col is not None:
    working_df[ctr_col] = numeric_series(working_df[ctr_col], percentage=True)

if updated_col is not None:
    working_df[updated_col] = pd.to_datetime(
        working_df[updated_col],
        errors="coerce",
    )

columns_to_keep = [
    column
    for column in [
        page_col,
        clicks_col,
        impressions_col,
        ctr_col,
        position_col,
        updated_col,
    ]
    if column is not None
]

page_df = working_df[columns_to_keep].copy()

if page_df[page_col].duplicated().any():
    aggregation_rules = {}

    for column in page_df.columns:
        if column == page_col:
            continue

        if column in [clicks_col, impressions_col]:
            aggregation_rules[column] = "sum"
        elif column == updated_col:
            aggregation_rules[column] = "max"
        else:
            aggregation_rules[column] = "mean"

    page_df = (
        page_df
        .groupby(page_col, as_index=False)
        .agg(aggregation_rules)
    )

    if clicks_col is not None and impressions_col is not None:
        page_df["calculated_ctr"] = np.where(
            page_df[impressions_col] > 0,
            page_df[clicks_col] / page_df[impressions_col],
            np.nan,
        )
        ctr_col = "calculated_ctr"

print(f"Unit of analysis: one content page/URL stored in the '{page_col}' column.")
print("Page-level dataframe shape:", page_df.shape)
print("Unique content pages:", page_df[page_col].nunique())
print("Missing page identifiers:", page_df[page_col].isna().sum())

display(page_df.head(10))

### Unit of Analysis

The unit of analysis is **one content page or URL**.

Each row in `page_df` represents one page that can be reviewed by the content team. The available columns provide signals about the current visibility and performance of that page.

Where the original starter data contained repeated rows for the same page, the records were aggregated into one page-level row. Clicks and impressions were summed, while rate or position values were averaged.

This unit is appropriate because the final decision is made at page level: the content editor decides which individual page should be reviewed or refreshed first.

### Sketching the Target and Proxy Columns

The true target, `target_incremental_clicks_28d`, cannot be filled yet because the pages have not been refreshed and observed for 28 days.

To make the framing concrete, I will create a temporary proxy score using the signals that are currently available.

The proxy is intentionally transparent. It is not presented as a trained machine learning model. Its purpose is to show what the target structure and resulting editorial queue could look like before model development begins.

In [ ]:
score_components = {}

if impressions_col is not None and impressions_col in page_df.columns:
    score_components["visibility"] = page_df[impressions_col].rank(pct=True)

if ctr_col is not None and ctr_col in page_df.columns:
    score_components["ctr_gap"] = 1 - page_df[ctr_col].rank(pct=True)

if position_col is not None and position_col in page_df.columns:
    position_values = pd.to_numeric(page_df[position_col], errors="coerce")
    score_components["ranking_potential"] = (
        1 - ((position_values - 10).abs() / 30)
    ).clip(lower=0, upper=1)

if updated_col is not None and updated_col in page_df.columns:
    reference_date = pd.Timestamp.today().normalize()
    age_in_days = (
        reference_date - page_df[updated_col]
    ).dt.days.clip(lower=0)

    score_components["staleness"] = age_in_days.rank(pct=True)

if not score_components:
    raise ValueError(
        "No suitable columns were found for the proxy. "
        "Map at least one impressions, CTR, position, or update-date column."
    )

component_df = pd.DataFrame(score_components).fillna(0.5)

preferred_weights = {
    "visibility": 0.35,
    "ctr_gap": 0.30,
    "ranking_potential": 0.20,
    "staleness": 0.15,
}

available_weights = {
    signal: preferred_weights[signal]
    for signal in component_df.columns
}

weight_total = sum(available_weights.values())

available_weights = {
    signal: weight / weight_total
    for signal, weight in available_weights.items()
}

page_df["proxy_refresh_opportunity_score"] = 0.0

for signal, weight in available_weights.items():
    page_df["proxy_refresh_opportunity_score"] += (
        component_df[signal] * weight
    )

page_df["proxy_refresh_opportunity_score"] = (
    page_df["proxy_refresh_opportunity_score"] * 100
).round(1)

page_df["target_incremental_clicks_28d"] = pd.Series(
    pd.NA,
    index=page_df.index,
    dtype="Float64",
)

print("Signals used in the proxy:")
print(list(component_df.columns))

print("\nNormalized signal weights:")
print(available_weights)

In [ ]:
preview_columns = [
    column
    for column in [
        page_col,
        impressions_col,
        clicks_col,
        ctr_col,
        position_col,
        updated_col,
        "proxy_refresh_opportunity_score",
        "target_incremental_clicks_28d",
    ]
    if column is not None and column in page_df.columns
]

target_preview = (
    page_df[preview_columns]
    .sort_values("proxy_refresh_opportunity_score", ascending=False)
    .head(10)
)

print(
    "The true target is currently empty because "
    "post-refresh outcome data has not yet been collected."
)

display(target_preview)

### Action Supported by the Output

The output will create a prioritized content-review queue.

Pages with high opportunity scores will be reviewed first for possible refresh or expansion. Medium-priority pages will be considered after the highest-priority opportunities. Low-priority pages may be protected or monitored.

The score does not directly decide that a page must be changed or removed. A content editor must still examine search intent, business relevance, factual accuracy, brand risk, and the quality of the available evidence.

In [ ]:
score_percentile = (
    page_df["proxy_refresh_opportunity_score"]
    .rank(pct=True, method="average")
)

page_df["priority_band"] = np.select(
    [
        score_percentile >= 0.67,
        score_percentile >= 0.34,
    ],
    [
        "High",
        "Medium",
    ],
    default="Low",
)

action_mapping = {
    "High": "Review first for refresh or expansion",
    "Medium": "Review after high-priority pages",
    "Low": "Protect or monitor; do not remove automatically",
}

page_df["supported_content_action"] = (
    page_df["priority_band"].map(action_mapping)
)

action_view_columns = [
    page_col,
    "proxy_refresh_opportunity_score",
    "priority_band",
    "supported_content_action",
    "target_incremental_clicks_28d",
]

ranked_review_queue = (
    page_df[action_view_columns]
    .sort_values("proxy_refresh_opportunity_score", ascending=False)
    .reset_index(drop=True)
)

display(ranked_review_queue.head(10))

## 5) Why ML Beats a Fixed Rule Here

A fixed rule might say:

> Review every page with more than 1,000 impressions and a CTR below 2%.

This rule is easy to understand, but it has several limitations.

First, content performance depends on multiple interacting factors. A page may have low CTR because of its search position, search intent, topic, content age, title, or competition. A single threshold cannot represent all of these situations.

Second, the same threshold may not be suitable for every page type. A product page, guide, news article, and landing page may have different normal performance patterns.

Third, fixed thresholds do not improve automatically when new examples and outcomes become available.

A machine learning model could learn relationships between several page features and actual post-refresh gains. It could also produce a continuous score, making it easier to rank limited editorial resources.

Human review is still required because the model may not understand business priorities, factual risks, brand requirements, or whether a page should remain unchanged.

### Current Limitations

This notebook frames the problem but does not train a production machine learning model.

The current opportunity score is a transparent proxy based on available signals. Its weights are provisional and have not yet been validated against actual post-refresh outcomes.

Possible limitations include missing or incomplete page data, inaccurate update dates, differences between content types, seasonal search demand, changes in search rankings, limited historical refresh records, and the possibility that high visibility does not always mean high business value.

The next stage should collect refresh decisions and their later outcomes. These outcomes can be used to train and evaluate a genuine scoring or ranking model.

## 6) Self-Check

- [x] I named the ML task type.
- [x] I defined the ideal target.
- [x] I created and explained a temporary proxy.
- [x] I selected a success metric.
- [x] I showed the unit of analysis as a real DataFrame.
- [x] I explained what one row represents.
- [x] I sketched the target column.
- [x] I connected the output to a real content action.
- [x] I explained why ML is more suitable than a fixed rule.
- [x] I documented the limitations and need for human review.

### AI Assistance Statement

I used an AI assistant to explore possible task-framing structures and to support the initial Python code organization.

I reviewed the proposed framing and selected the lane, task type, target, proxy, success metric, unit of analysis, content action, and limitations based on my understanding of the assignment.

The final decisions and explanations in this notebook represent my own framing of the problem.

In [ ]:
self_check = pd.DataFrame({
    "Requirement": [
        "ML task type named",
        "Target or proxy defined",
        "Success metric selected",
        "Real DataFrame displayed",
        "Unit of analysis explained",
        "Target column sketched",
        "ML versus fixed rule explained",
        "Output tied to content action",
    ],
    "Completed": [
        True,
        True,
        True,
        not page_df.empty,
        page_col is not None,
        "target_incremental_clicks_28d" in page_df.columns,
        True,
        "supported_content_action" in page_df.columns,
    ],
})

display(self_check)

assert self_check["Completed"].all(), (
    "One or more notebook requirements are incomplete."
)

print("Self-check passed. The notebook is ready to save and commit.")